# 01 — Greedy Algorithms

## Why This Matters for DSA
A **Greedy Algorithm** is an algorithmic paradigm that builds up a solution piece by piece, always choosing the next piece that offers the most obvious and immediate benefit. In technical terms, it makes the **locally optimal choice** at each step with the hope of finding a **globally optimal solution**.

Unlike Dynamic Programming (which explores all options and remembers subproblem solutions) or Backtracking (which searches the entire solution space and backtracks upon failure), Greedy algorithms are typically simpler and faster. They usually run in $O(N \log N)$ (due to sorting) or $O(N)$ time, requiring very little auxiliary space.

Greedy algorithms are crucial in computer science for solving network optimization problems (like Minimum Spanning Trees, Dijkstra's Shortest Path), data compression (Huffman Coding), and various scheduling and allocation problems.

## Prerequisites
- `01_Foundations` (Phase 01) — Time complexity ($O$-notation).
- `01_Heaps_and_Priority_Queues` (Phase 03) — Min/Max heap properties and usage.
- Standard sorting algorithms and custom comparators in C++.

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain the **Greedy Choice Property** and **Optimal Substructure**.
- Design and implement the **Fractional Knapsack Problem**.
- Prove the optimality of sorting by finish times in **Interval Scheduling (Activity Selection)**.
- Construct a **Huffman Coding Tree** using priority queues for optimal prefix-free codes.
- Distinguish when Greedy algorithms fail (e.g., 0/1 Knapsack) and when they succeed.
- Verify the performance profiles of Greedy vs. Dynamic Programming paradigms empirically.

## Checklist Before Moving On
- [ ] I can explain the difference between Greedy, Dynamic Programming, and Backtracking.
- [ ] I can write the Fractional Knapsack algorithm in C++ and explain why it does not work for the 0/1 Knapsack problem.
- [ ] I can trace and implement the Activity Selection problem.
- [ ] I can build a Huffman tree and generate prefix-free variable-length binary codes.
- [ ] I can solve job sequencing problems with deadlines and profits using greedy strategy.

## Table of Contents
1. Core Principles of Greedy Algorithms
2. Fractional Knapsack Problem — Densest-First Strategy
3. Interval Scheduling (Activity Selection)
4. Huffman Coding — Greedy Tree Merge
5. Benchmarking: Greedy (Fractional) vs. Dynamic Programming (0/1 Knapsack)
6. Checkpoints, Mini Project, and Answer Key
7. LeetCode Practice Problems

## 1. Core Principles of Greedy Algorithms

### The Intuition
Imagine you are walking down a path and at each intersection, you see coins of different values. A greedy approach says: **"At every junction, take the largest coin available right now, and keep going forward."** You never look back, and you never look further ahead than the immediate options. 

This local strategy is highly efficient: you make decisions quickly without planning complex schedules. However, this strategy does not always lead to the maximum total money if taking a smaller coin now opens up a path to a massive chest of gold later.

### Core Requirements for Optimality
For a greedy algorithm to produce a **globally optimal** solution, the problem must possess two key mathematical properties:

1. **Greedy Choice Property:** A global optimal solution can be arrived at by making locally optimal (greedy) choices. In other words, we can make the choice that looks best in the current situation without ever having to reconsider it.
2. **Optimal Substructure:** A problem exhibits optimal substructure if an optimal solution to the entire problem contains within it optimal solutions to its subproblems. For example, if the shortest path from $A$ to $C$ passes through $B$, then the subpath from $A$ to $B$ must also be the shortest path.

### How Do We Prove a Greedy Algorithm is Correct?
Proving correctness mathematically is vital because intuitive greedy strategies frequently fail. The two main proof techniques are:
- **Greedy Stays Ahead:** Show that at each step of the algorithm, the greedy choice makes progress that is at least as good as any other potential choice (e.g., in Activity Selection, showing that greedy always finishes the $k$-th activity no later than any other algorithm).
- **Exchange Argument:** Show that any optimal solution can be gradually transformed (by exchanging elements) into the greedy solution without degrading its quality.

## 2. Fractional Knapsack Problem — Densest-First Strategy

### The Why
In the **Fractional Knapsack** problem, we are given a set of $N$ items. Each item $i$ has a weight $w_i$ and a value $v_i$. We have a knapsack of capacity $W$. Our goal is to maximize the total value in our knapsack, and we are allowed to take fractions of items (e.g., breaking a bar of gold, taking spoonfuls of spices).

Since we can take fractions, we should prioritize the items that give us the most "bang for our buck" — the highest **value density**: 
$$\text{Density} = \frac{v_i}{w_i}$$

### Core Mechanism
1. Calculate the value-to-weight ratio (density) for each item.
2. Sort all items in descending order of their density.
3. Iterate through the sorted list:
   - If the item's weight fits completely in the remaining capacity, add the item fully.
   - If the item's weight exceeds the remaining capacity, add a fraction of the item to fill the knapsack completely and terminate.

### Complexity
- **Time Complexity:** $O(N \log N)$ to sort the items. The greedy loop itself runs in $O(N)$ time.
- **Space Complexity:** $O(N)$ to store items (or $O(1)$ auxiliary space if sorting is done in-place).

Let's write and execute the C++ implementation.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>
#include <iomanip>

using namespace std;

struct Item {
    int id;
    double weight;
    double value;
    
    // Calculate density
    double getDensity() const {
        return value / weight;
    }
};

// Comparator to sort items by density in descending order
bool compareItems(const Item& a, const Item& b) {
    return a.getDensity() > b.getDensity();
}

struct SelectionResult {
    int id;
    double fraction;
    double valueTaken;
    double weightTaken;
};

pair<double, vector<SelectionResult>> getFractionalKnapsack(double capacity, vector<Item>& items) {
    // 1. Sort items by density
    sort(items.begin(), items.end(), compareItems);
    
    double totalValue = 0.0;
    vector<SelectionResult> selections;
    
    for (const auto& item : items) {
        if (capacity <= 0) break;
        
        if (item.weight <= capacity) {
            // Take the whole item
            capacity -= item.weight;
            totalValue += item.value;
            selections.push_back({item.id, 1.0, item.value, item.weight});
        } else {
            // Take fraction of the item to fill the remaining capacity
            double fraction = capacity / item.weight;
            double valTaken = item.value * fraction;
            totalValue += valTaken;
            selections.push_back({item.id, fraction, valTaken, capacity});
            capacity = 0; // Knapsack is full
            break;
        }
    }
    return {totalValue, selections};
}

int main() {
    cout << "--- Fractional Knapsack Demo ---\n";
    
    vector<Item> items = {
        {1, 10, 60},  // Density = 6.0
        {2, 20, 100}, // Density = 5.0
        {3, 30, 120}  // Density = 4.0
    };
    double W = 50.0;
    
    auto result = getFractionalKnapsack(W, items);
    double maxValue = result.first;
    auto selections = result.second;
    
    cout << fixed << setprecision(2);
    cout << "Knapsack Capacity: " << W << " kg\n";
    cout << "Maximum Value Obtained: $" << maxValue << "\n\n";
    cout << "Selected Items Breakdown:\n";
    for (const auto& sel : selections) {
        cout << "  Item " << sel.id 
             << " (Fraction: " << sel.fraction * 100 << "%, Weight taken: " 
             << sel.weightTaken << " kg, Value: $" << sel.valueTaken << ")\n";
    }
    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Interval Scheduling (Activity Selection)

### The Why
You have a single resource (e.g., a lecture hall, a meeting room, or a CPU core) and a set of $N$ activities. Each activity $i$ has a start time $s_i$ and a finish time $f_i$. Two activities are compatible if their time intervals do not overlap. Our goal is to select the **maximum number** of mutually compatible activities.

### Exploring Greedy Strategies: Which One Wins?
When designing a greedy algorithm, we need to decide what metric to optimize. Let's analyze candidate metrics:

1. **Sort by Start Time:** Pick the activity that starts earliest.
   - *Why it fails:* An activity that starts extremely early but lasts all day will prevent all other activities from scheduling. (Counterexample: $[0, 24]$ vs $[1, 2], [2, 3]$).
2. **Sort by Shortest Duration:** Pick the activity with the smallest duration ($f_i - s_i$).
   - *Why it fails:* A short activity might sit right in the middle, overlapping with two longer but mutually compatible activities. (Counterexample: $[1, 5], [4, 6], [5, 9]$ - duration of middle is 2, picking it blocks both $[1, 5]$ and $[5, 9]$).
3. **Sort by Fewest Overlaps:** Pick the activity that conflicts with the fewest other activities.
   - *Why it fails:* Complex configurations can be constructed where choosing the one with fewer overlaps leads to a suboptimal count.
4. **Sort by Finish Time:** Pick the activity that finishes earliest.
   - *Why it is optimal:* Finishing an activity as early as possible **leaves the maximum possible room** for subsequent activities. This greedy choice "stays ahead" of any other choice.

### Core Mechanism
1. Sort all activities by their finish times in non-decreasing order.
2. Select the first activity from the sorted list.
3. For the remaining activities, if the start time of the next activity is greater than or equal to the finish time of the last selected activity, select it.
4. Repeat until all activities are processed.

```
Timeline of Sorted Activities:
Act 1: [1, 4]  =======> Select Act 1 (finishes at 4)
Act 2: [3, 5]  Overlaps with Act 1 (start 3 < finish 4) -> Skip
Act 3: [0, 6]  Overlaps with Act 1 (start 0 < finish 4) -> Skip
Act 4: [5, 7]  =======> Select Act 4 (start 5 >= finish 4)
Act 5: [8, 9]  =======> Select Act 5 (start 8 >= finish 7)
```

### Complexity
- **Time Complexity:** $O(N \log N)$ to sort activities. The selection pass takes $O(N)$ time.
- **Space Complexity:** $O(1)$ auxiliary space if sorting is done in-place.

Let's write and execute the C++ implementation.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

struct Activity {
    int id;
    int start;
    int finish;
};

// Sort by finish time. If finish times are equal, sort by start time descending.
bool compareActivities(const Activity& a, const Activity& b) {
    if (a.finish == b.finish) {
        return a.start > b.start;
    }
    return a.finish < b.finish;
}

vector<Activity> selectActivities(vector<Activity>& activities) {
    if (activities.empty()) return {};
    
    // 1. Sort by finish times
    sort(activities.begin(), activities.end(), compareActivities);
    
    vector<Activity> selected;
    
    // 2. Greedily select the first activity
    selected.push_back(activities[0]);
    int lastFinish = activities[0].finish;
    
    // 3. Scan remaining activities
    for (size_t i = 1; i < activities.size(); ++i) {
        if (activities[i].start >= lastFinish) {
            selected.push_back(activities[i]);
            lastFinish = activities[i].finish;
        }
    }
    return selected;
}

int main() {
    cout << "--- Activity Selection Demo ---\n";
    
    vector<Activity> activities = {
        {1, 5, 9},
        {2, 1, 2},
        {3, 3, 4},
        {4, 0, 6},
        {5, 5, 7},
        {6, 8, 9}
    };
    
    vector<Activity> result = selectActivities(activities);
    
    cout << "Maximum number of activities scheduled: " << result.size() << "\n";
    cout << "Scheduled Activities:\n";
    for (const auto& act : result) {
        cout << "  Activity " << act.id << " : [" << act.start << ", " << act.finish << "]\n";
    }
    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Huffman Coding — Greedy Tree Merge

### The Why
Data transmission and storage benefit immensely from compression. If we represent every character in a text with a fixed number of bits (like ASCII, which uses 8 bits per character), we waste space because common letters (like 'e' or 'a') occupy the same space as rare letters (like 'z' or 'q').

**Huffman Coding** is a lossless data compression technique that uses variable-length codes:
- Frequent characters get short codes (fewer bits).
- Rare characters get long codes (more bits).

To make these codes decodable without ambiguity, they must be **prefix-free** (no code is a prefix of another code). For example, if 'A' is `0` and 'B' is `01`, this is NOT prefix-free because upon reading `0`, we don't know if it's 'A' or the beginning of 'B'. Huffman codes solve this by placing all character symbols strictly at the **leaves** of a binary tree.

### Core Mechanism
Building the optimal tree uses a greedy tree merging strategy:
1. Create a leaf node for each character and add it to a min-priority queue (sorted by character frequency).
2. While there is more than one node in the queue:
   - Pop the two nodes with the lowest frequencies. These are the two least frequent elements.
   - Create a new internal node with a frequency equal to the sum of the two nodes' frequencies.
   - Make the first popped node its left child, and the second popped node its right child.
   - Push this new internal node back into the min-priority queue.
3. The remaining single node in the queue is the root of the Huffman tree.
4. Traverse the tree from the root to the leaves: assign `0` for left branches and `1` for right branches to form the binary code for each character.

### Complexity
- **Time Complexity:** $O(N \log N)$ where $N$ is the number of unique characters. Each priority queue push/pop takes $O(\log N)$, executed $2N-2$ times.
- **Space Complexity:** $O(N)$ to store the tree nodes.

Let's write and execute the C++ implementation.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <unordered_map>
#include <string>

using namespace std;

// Tree node representation
struct HuffmanNode {
    char character;
    int frequency;
    HuffmanNode *left, *right;
    
    HuffmanNode(char ch, int freq) : character(ch), frequency(freq), left(nullptr), right(nullptr) {}
    
    ~HuffmanNode() {
        delete left;
        delete right;
    }
};

// Custom comparator for min-priority queue
struct CompareNodes {
    bool operator()(HuffmanNode* const& a, HuffmanNode* const& b) {
        return a->frequency > b->frequency;
    }
};

HuffmanNode* buildHuffmanTree(const vector<char>& data, const vector<int>& freq) {
    priority_queue<HuffmanNode*, vector<HuffmanNode*>, CompareNodes> pq;
    
    // 1. Initialize leaf nodes and add to priority queue
    for (size_t i = 0; i < data.size(); ++i) {
        pq.push(new HuffmanNode(data[i], freq[i]));
    }
    
    // 2. Greedily merge nodes
    while (pq.size() > 1) {
        HuffmanNode* left = pq.top(); pq.pop();
        HuffmanNode* right = pq.top(); pq.pop();
        
        // Internal node has dummy character '\0'
        HuffmanNode* merged = new HuffmanNode('\0', left->frequency + right->frequency);
        merged->left = left;
        merged->right = right;
        
        pq.push(merged);
    }
    
    return pq.top();
}

// Traverse tree to extract prefix-free codes
void generateCodes(HuffmanNode* root, const string& code, unordered_map<char, string>& codesMap) {
    if (!root) return;
    
    // If it's a leaf node, store the code
    if (!root->left && !root->right) {
        codesMap[root->character] = code;
        return;
    }
    
    generateCodes(root->left, code + "0", codesMap);
    generateCodes(root->right, code + "1", codesMap);
}

int main() {
    cout << "--- Huffman Coding Demo ---\n";
    
    vector<char> characters = {'a', 'b', 'c', 'd', 'e', 'f'};
    vector<int> frequencies = {5, 9, 12, 13, 16, 45};
    
    HuffmanNode* root = buildHuffmanTree(characters, frequencies);
    
    unordered_map<char, string> huffmanCodes;
    generateCodes(root, "", huffmanCodes);
    
    cout << "Character Codes:\n";
    for (char ch : characters) {
        cout << "  " << ch << " : " << huffmanCodes[ch] << "\n";
    }
    
    // Clean up memory
    delete root;
    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Benchmarking: Greedy Choice vs. Dynamic Programming

### The Why
To understand the power and limitations of the Greedy approach, we must compare it to Dynamic Programming on the same problem style. The Knapsack problem serves as a perfect lens:

| Feature | Fractional Knapsack (Greedy) | 0/1 Knapsack (Dynamic Programming) |
|---|---|---|
| **Decision Constraint** | Can break items. | Items are indivisible (take or leave). |
| **Optimal Paradigm** | Greedy choice yields global optimum. | Greedy fails; requires Dynamic Programming. |
| **Time Complexity** | $O(N \log N)$ (sorting) | $O(N \cdot W)$ (pseudo-polynomial) |
| **Space Complexity** | $O(1)$ auxiliary | $O(W)$ (space-optimized DP array) |

### Why Greedy Fails for 0/1 Knapsack
Suppose our capacity is $W = 50$, and we have:
- Item A: weight = 10, value = 60 (density = 6)
- Item B: weight = 20, value = 100 (density = 5)
- Item C: weight = 30, value = 120 (density = 4)

**If we must choose greedily (highest density first) without breaking items (0/1):**
1. We pick Item A (capacity remaining = 40, value = 60).
2. We pick Item B (capacity remaining = 20, value = 160).
3. Item C (weight 30) does not fit anymore. We stop. Total value = **160**.

**Optimal solution (found via Dynamic Programming):**
- Pick Item B and Item C (weight = 20 + 30 = 50, value = 100 + 120 = **220**).

Let's compile and run an empirical benchmark to compare the execution speed of the Greedy Fractional Knapsack algorithm against the Dynamic Programming 0/1 Knapsack algorithm.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>
#include <chrono>
#include <random>

using namespace std;

struct Item {
    double weight;
    double value;
    
    double getDensity() const {
        return value / weight;
    }
};

bool compareItems(const Item& a, const Item& b) {
    return a.getDensity() > b.getDensity();
}

// Greedy Fractional Knapsack: O(N log N)
double solveFractionalKnapsackGreedy(double capacity, vector<Item> items) {
    sort(items.begin(), items.end(), compareItems);
    double totalValue = 0.0;
    for (const auto& item : items) {
        if (capacity <= 0) break;
        if (item.weight <= capacity) {
            capacity -= item.weight;
            totalValue += item.value;
        } else {
            totalValue += item.value * (capacity / item.weight);
            capacity = 0;
            break;
        }
    }
    return totalValue;
}

// 0/1 Knapsack via Dynamic Programming (Space optimized 1D array): O(N * W)
int solve01KnapsackDP(int capacity, const vector<Item>& items) {
    int n = items.size();
    // dp[w] stores maximum value with capacity w
    vector<int> dp(capacity + 1, 0);
    
    for (int i = 0; i < n; ++i) {
        int w = (int)items[i].weight;
        int v = (int)items[i].value;
        // Traverse backwards to avoid using the same item multiple times
        for (int j = capacity; j >= w; --j) {
            dp[j] = max(dp[j], dp[j - w] + v);
        }
    }
    return dp[capacity];
}

int main() {
    const int N = 3000;
    const int W = 5000;
    
    mt19937 rng(1337);
    uniform_int_distribution<int> distW(1, 100);   // weights between 1 and 100
    uniform_int_distribution<int> distV(10, 1000); // values between 10 and 1000
    
    vector<Item> items(N);
    for (int i = 0; i < N; ++i) {
        items[i] = { (double)distW(rng), (double)distV(rng) };
    }
    
    cout << "--- Benchmarking Greedy vs Dynamic Programming ---\n";
    cout << "Problem Size: N = " << N << " items, Capacity W = " << W << "\n\n";
    
    // 1. Benchmark Greedy Fractional Knapsack
    {
        auto start = chrono::high_resolution_clock::now();
        double greedyVal = solveFractionalKnapsackGreedy(W, items);
        auto end = chrono::high_resolution_clock::now();
        auto duration = chrono::duration_cast<chrono::microseconds>(end - start).count();
        
        cout << "[Greedy Fractional Knapsack]\n";
        cout << "  Max Value: " << greedyVal << "\n";
        cout << "  Time: " << duration << " microseconds\n\n";
    }
    
    // 2. Benchmark DP 0/1 Knapsack
    {
        auto start = chrono::high_resolution_clock::now();
        int dpVal = solve01KnapsackDP(W, items);
        auto end = chrono::high_resolution_clock::now();
        auto duration = chrono::duration_cast<chrono::microseconds>(end - start).count();
        
        cout << "[DP 0/1 Knapsack]\n";
        cout << "  Max Value: " << dpVal << "\n";
        cout << "  Time: " << duration << " microseconds\n\n";
    }
    
    cout << "Conclusion: Greedy Fractional is dramatically faster because sorting N elements takes O(N log N) time, "
         << "whereas 0/1 Knapsack Dynamic Programming requires scanning a table of size N * W. "
         << "However, Greedy ONLY works when fractional item selection is allowed!\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Checkpoint, Mini Project, and Answer Key

### Checkpoint Questions
1. **Why does the greedy choice fail for the 0/1 Knapsack problem but work for Fractional Knapsack?**
2. **In the Activity Selection problem, if you sort by start time, why is it suboptimal? Provide a concrete counterexample.**
3. **What is the worst-case space complexity of building a Huffman coding tree, and why?**
4. **Under what conditions is a greedy algorithm guaranteed to yield a globally optimal solution?**

### Answer Key
1. In 0/1 Knapsack, the indivisibility of items prevents us from fully utilizing the remaining capacity of the knapsack. Choosing a high-density item can block a combination of slightly lower-density items that together pack more value into the same capacity. In Fractional Knapsack, because we can partition items, we never waste capacity, allowing the density sorting strategy to be optimal.
2. Sorting by start time can lead us to select an activity that starts early but has a very long duration, blocking many shorter, compatible activities. Counterexample: Activities A: $[0, 20]$, B: $[1, 2]$, C: $[2, 3]$. Sorting by start time schedules A, blocking both B and C (yields 1 activity scheduled instead of 2).
3. The worst-case space complexity is $O(N)$ because for $N$ unique characters, we store $N$ leaf nodes and $N-1$ internal nodes in memory. The height of the tree does not exceed $N$ (skewed tree), but the absolute node count remains $2N-1$.
4. A greedy algorithm yields a globally optimal solution only if the problem satisfies both the **Greedy Choice Property** (a local optimum selection can never trap the solver into a worse state) and the **Optimal Substructure** property (an optimal solution contains optimal subsolutions).

---

### Mini Project: Task Scheduler with Deadlines and Profits (Job Sequencing)
A classic problem is: You have a list of $N$ jobs. Each job $i$ has a job ID, a deadline $d_i$, and a profit $p_i$. Each job takes exactly 1 unit of time to complete. A job is completed if it starts before its deadline. Only one job can be executed at a time. The goal is to maximize total profit.

**Greedy Strategy:**
1. Sort all jobs in descending order of profit.
2. Find the maximum deadline among all jobs. Create slots up to this deadline.
3. Iterate through the sorted jobs:
   - Try to schedule the job in its latest possible available slot (starting from its deadline and searching backwards to slot 1).
   - Scheduling a job as late as possible leaves earlier slots open for other jobs with tighter deadlines.
   - If no slot is free before its deadline, skip the job.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <numeric>

using namespace std;

struct Job {
    char id;
    int deadline;
    int profit;
};

// Sort jobs by profit descending
bool compareJobs(const Job& a, const Job& b) {
    return a.profit > b.profit;
}

pair<int, vector<char>> scheduleJobs(vector<Job>& jobs) {
    // 1. Sort by profit descending
    sort(jobs.begin(), jobs.end(), compareJobs);
    
    // Find maximum deadline to determine size of schedule array
    int maxDeadline = 0;
    for (const auto& job : jobs) {
        maxDeadline = max(maxDeadline, job.deadline);
    }
    
    // schedule[i] stores the job ID scheduled in slot i. 0 means empty.
    // Time slots are 1-indexed up to maxDeadline
    vector<char> schedule(maxDeadline + 1, '0');
    int totalProfit = 0;
    
    // 2. Greedily place jobs in their latest possible slots
    for (const auto& job : jobs) {
        for (int slot = job.deadline; slot >= 1; --slot) {
            if (schedule[slot] == '0') {
                schedule[slot] = job.id;
                totalProfit += job.profit;
                break; // Job scheduled, move to next
            }
        }
    }
    
    // Collect scheduled job IDs in sequence
    vector<char> jobSequence;
    for (int slot = 1; slot <= maxDeadline; ++slot) {
        if (schedule[slot] != '0') {
            jobSequence.push_back(schedule[slot]);
        }
    }
    
    return {totalProfit, jobSequence};
}

int main() {
    cout << "--- Task Scheduler with Deadlines and Profits ---\n";
    
    vector<Job> jobs = {
        {'A', 2, 100},
        {'B', 1, 19},
        {'C', 2, 27},
        {'D', 1, 25},
        {'E', 3, 15}
    };
    
    auto result = scheduleJobs(jobs);
    int profit = result.first;
    vector<char> sequence = result.second;
    
    cout << "Maximum Profit Earned: $" << profit << "\n";
    cout << "Optimal Job Sequence: ";
    for (char id : sequence) {
        cout << id << " ";
    }
    cout << "\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. LeetCode Practice Problems

| # | Problem | Difficulty | Topic | Hint |
|---|---|---|---|---|
| 435 | [Non-overlapping Intervals](https://leetcode.com/problems/non-overlapping-intervals/) | Medium | Interval Scheduling | Equivalent to Activity Selection. Find the maximum number of non-overlapping intervals, then return total intervals minus this maximum count. |
| 452 | [Minimum Number of Arrows to Burst Balloons](https://leetcode.com/problems/minimum-number-of-arrows-to-burst-balloons/) | Medium | Interval Overlap | Sort balloons by finish coordinate. Place an arrow at the finish coordinate of the first balloon, and skip all overlapping balloons that start before or at this coordinate. |
| 55 | [Jump Game](https://leetcode.com/problems/jump-game/) | Medium | Greedy Reachability | Track the maximum reachable index from current index. If at any index $i$ the maximum reachable is less than $i$, return false. If maximum reachable reaches or exceeds the last index, return true. |
| 134 | [Gas Station](https://leetcode.com/problems/gas-station/) | Medium | Greedy Cycle | If total gas is less than total cost, a solution is impossible. Otherwise, start at 0, track current tank gas. If tank falls below 0, reset starting station to next index and reset tank to 0. |
| 406 | [Queue Reconstruction by Height](https://leetcode.com/problems/queue-reconstruction-by-height/) | Medium | Greedy Insertion | Sort people by height descending; if height is equal, sort by count ascending. Then insert each person into a list at the index specified by their count. |

### Summary of Phase 05 — Greedy Algorithms
Congratulations! You have completed the first notebook of **Phase 05 (Algorithmic Paradigms)**:
1. You mastered the properties of **Greedy Choice Property** and **Optimal Substructure**.
2. You implemented and verified **Fractional Knapsack** (sorting by density).
3. You implemented and proved **Activity Selection** (sorting by finish time).
4. You built a **Huffman Coding** compression tree using priority queues.
5. You analyzed the difference between Greedy and Dynamic Programming empirically and solved the **Job Sequencing** mini-project.

In the next notebook, we move on to **02_Dynamic_Programming_Foundations.ipynb**, diving deep into memoization, tabulation, and the core structural state formulations of Dynamic Programming.